# Bi-Mamba 55M — Chinese ↔ Vietnamese translation, end-to-end on Colab

Notebook chạy **toàn bộ pipeline** từ A đến Z: tải dữ liệu → train tokenizer → train Bi-Mamba 55M → đánh giá SacreBLEU → dịch demo.

**Yêu cầu:** Runtime → GPU (T4 / L4 / A100 đều OK). RAM ≥ 12 GB.

> Mặc định subsample còn 200 K cặp để chạy đầy đủ trên T4 free trong khoảng 1.5–2 giờ. Bỏ subsample nếu muốn train trên toàn bộ corpus.


## 1. Mount Google Drive (tuỳ chọn)

Để lưu checkpoint vĩnh viễn. Nếu không cần, bạn có thể bỏ qua ô này.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Skipping drive mount:', e)


## 2. Clone repo

Đang clone từ **`ChauDucToan/NLP_DHM`** — đổi `REPO_URL` nếu bạn fork sang chỗ khác.

In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
REPO_DIR = '/content/NLP_DHM'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only || true
%cd $REPO_DIR
!pwd && ls


## 3. Cài đặt dependencies cơ bản

Cài các package thuần Python (`torch`, `sentencepiece`, `sacrebleu`, ...). CUDA fast-path cho Mamba sẽ được cài ở ô **3b** ngay bên dưới.


In [ ]:
!pip install -q -r requirements.txt
import torch, sys
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
sys.path.insert(0, 'src')


### 3b. Tăng tốc Mamba bằng CUDA kernels (tuỳ chọn nhưng khuyên dùng)

Cài wheel **trực tiếp từ GitHub Releases** thay vì build từ PyPI (PyPI sẽ cố compile và hầu như fail trên Colab vì lệch ABI / CUDA toolkit). Mất ~30 giây. Train sẽ nhanh ~5–10× nếu thành công, fallback PyTorch vẫn chạy đúng nếu fail.


In [ ]:
# === Optional but recommended: CUDA fast-path (5–10× speedup) ===
# `pip install mamba-ssm causal-conv1d` from PyPI tries to *build* from source
# on Colab and almost always fails (CUDA toolkit / torch ABI mismatch).
# Instead, install matching prebuilt wheels straight from the GitHub releases.
import sys, subprocess, torch

# Skip on CPU runtimes (no benefit, and the wheels need CUDA).
if not torch.cuda.is_available():
    print('No CUDA → skipping fast-path install (pure-PyTorch fallback will run).')
else:
    torch_mm  = '.'.join(torch.__version__.split('+')[0].split('.')[:2])  # e.g. '2.10'
    cuda_maj  = torch.version.cuda.split('.')[0]                          # e.g. '12'
    py_tag    = f'cp{sys.version_info.major}{sys.version_info.minor}'     # e.g. 'cp312'
    abi       = 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
    suffix    = f'cu{cuda_maj}torch{torch_mm}cxx11abi{abi}-{py_tag}-{py_tag}-linux_x86_64.whl'

    ccv1d_ver = '1.6.1'
    mamba_ver = '2.3.1'
    ccv1d_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{ccv1d_ver}.post4/causal_conv1d-{ccv1d_ver}+{suffix}'
    mamba_url = f'https://github.com/state-spaces/mamba/releases/download/v{mamba_ver}/mamba_ssm-{mamba_ver}+{suffix}'

    print('Installing causal-conv1d:', ccv1d_url.rsplit("/", 1)[-1])
    r1 = subprocess.run(['pip', 'install', '-q', '--no-deps', ccv1d_url])
    print('Installing mamba-ssm:    ', mamba_url.rsplit("/", 1)[-1])
    r2 = subprocess.run(['pip', 'install', '-q', '--no-deps', mamba_url])

    ok = True
    try:
        import causal_conv1d, mamba_ssm  # noqa
        from mamba_ssm.ops.selective_scan_interface import selective_scan_fn  # noqa
        from causal_conv1d import causal_conv1d_fn  # noqa
    except Exception as e:
        ok = False
        print('FAST-PATH INSTALL FAILED:', e)
        print(' → falling back to pure-PyTorch Mamba (training still works, ~5–10× slower).')
    if ok:
        print('CUDA fast-path active: selective_scan_fn + causal_conv1d_fn imported OK.')


## 4. Configuration

Sửa các giá trị bên dưới nếu muốn chạy nhanh / lâu hơn. Mặc định: subsample 200 K cặp, max_steps 30 K — phù hợp T4 trong 1.5–2 giờ.

In [ ]:
import os
import yaml
from pathlib import Path

cfg_path = Path('configs/bi_mamba_55m.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

# =====================================================================
# Data knobs
# =====================================================================
# `everyday` is the recommended default: TED2020 + WikiMatrix +
# OpenSubtitles (cap 80k) + NLLB (cap 50k) + bible-uedin (cap 6k) → ~225k
# clean pairs, bible only ~3% of mix. OpenCC Trad→Simp normalisation +
# Cantonese-particle filter make the zh side consistent Mandarin Simplified.
# `small` (old default) leaves bible at ~14% → biblical-register bias.
# `medium` (uncapped OpenSubtitles) collapses BLEU to ~1–2 and is NOT recommended.
cfg['data']['preset']           = 'everyday'    # tiny | small | everyday | medium | large
cfg['data']['max_train_pairs']  = 400_000       # set to None to use everything
cfg['data']['max_valid_pairs']  = 2_000
cfg['data']['max_test_pairs']   = 2_000
# Strict per-language script + length-ratio filter (drops WikiMatrix noise).
cfg['data']['script_check']     = True
cfg['data']['min_zh_vi_ratio']  = 0.10
cfg['data']['max_zh_vi_ratio']  = 1.20
# Dialect / script normalisation (added in PR #8). Without these flags the
# zh side is dialect-heterogeneous (TED2020 ~70% Cantonese in Traditional,
# bible-uedin Traditional, WikiMatrix mixed Trad/Simp/Classical, only
# OpenSubtitles vi-zh_cn is true Simplified). Set False to disable.
cfg['data']['zh_normalize_simplified'] = True   # OpenCC t2s on zh side
cfg['data']['zh_filter_cantonese']     = True   # drop pairs with 嘅/哋/啲/咁/喺/嗰/咗...
# Per-source caps. Apply only if the source is in the preset.
cfg['data']['max_pairs_per_source'] = {
    'opensubtitles': 80_000,
    'wikimatrix':    100_000,
    'nllb':          50_000,
    'bible_uedin':   6_000,
}
# NOTE: NLLB triggers a one-time ~700 MB download. Drop it from the preset
# (use `small`) if your Colab has limited disk / bandwidth.

# =====================================================================
# Tokenizer / model knobs
# =====================================================================
cfg['model']['vocab_size']      = 32_000        # 16K stays at ~55M params; 32K ≈ +0.5–1 BLEU
cfg['tokenizer']['vocab_size']  = cfg['model']['vocab_size']

# =====================================================================
# Train knobs — 120k steps × batch 128 ≈ 70 epochs over ~450k bi-direction
# pairs. Early-stopping (patience=15 ticks × 2k = 30k stale steps) makes
# this safe to leave high — training auto-stops when val_loss plateaus,
# typically around step 50–80k.
# =====================================================================
cfg['train']['max_steps']      = 120_000
cfg['train']['warmup_steps']   = 4_000
cfg['train']['batch_size']     = 128            # 64 if you OOM; 256 on A100
cfg['train']['grad_accum_steps'] = 1
cfg['train']['amp_dtype']      = 'bf16'         # 'fp16' on old T4 if bf16 unsupported
cfg['train']['eval_every']     = 2_000
cfg['train']['save_every']     = 2_000
cfg['train']['log_every']      = 50

# Auto-detect CPU count.
_n_cpu = os.cpu_count() or 4
cfg['train']['num_workers']    = min(8, max(2, _n_cpu - 2))
print(f'Using num_workers={cfg["train"]["num_workers"]} (detected {_n_cpu} CPUs)')

# Length-bucketed batching — same VRAM, ~1.5× more tokens/sec.
cfg['train']['length_bucket']      = True
cfg['train']['bucket_pool_factor'] = 100
# EMA of weights: +0.3–1.0 BLEU "free".
cfg['train']['ema']            = True
cfg['train']['ema_decay']      = 0.9990
# Early stopping (15 ticks × eval_every 2_000 = 30k steps stale → stop).
cfg['train']['early_stopping_patience']   = 15
cfg['train']['early_stopping_min_delta']  = 0.0

# =====================================================================
# Eval knobs
# =====================================================================
cfg['eval']['beam_size']        = 6
cfg['eval']['length_penalty']   = {'zh2vi': 1.20, 'vi2zh': 0.80}
cfg['eval']['num_samples']      = 1_000

Path('configs/_colab.yaml').write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True)
)
print('Saved configs/_colab.yaml')


## 5. Tải + chuẩn bị dữ liệu (OPUS zh-vi)

`prepare_data.py` tải trực tiếp từ **OPUS** (`object.pouta.csc.fi`). Không cần Hugging Face dataset config — `Helsinki-NLP/opus-100` thực ra **không có** cặp zh-vi.

Presets:

| preset      | sources                                                                              | ~ pairs   | download  |
|-------------|--------------------------------------------------------------------------------------|-----------|-----------|
| `tiny`      | TED2020                                                                              | 50 K      | 1 MB      |
| `small`     | TED2020 + WikiMatrix + bible-uedin (no caps)                                         | 200 K     | 25 MB     |
| `everyday` *(default)* | TED2020 + WikiMatrix + OpenSubtitles (cap 80k) + NLLB (cap 50k) + bible-uedin (cap 6k) | 225 K     | ~700 MB   |
| `medium`    | small + OpenSubtitles vi-zh_cn (uncapped, **NOT** recommended)                       | 3 M       | 65 MB     |
| `large`   | medium + NLLB                                            | 30 M      | 700 MB|

Bạn cũng có thể truyền `--custom-jsonl path/to/your.jsonl` (mỗi dòng `{"zh": "...", "vi": "..."}`).

In [ ]:
!python scripts/prepare_data.py --config configs/_colab.yaml --preset {cfg['data']['preset']}


## 6. Train SentencePiece tokenizer (chia sẻ chung zh+vi)

In [ ]:
!python scripts/train_tokenizer.py --config configs/_colab.yaml


## 7. Khởi tạo model + kiểm tra số tham số

Mục tiêu thiết kế: ~55 M tham số. Thực tế: **54.63 M**.

In [ ]:
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.utils import count_parameters, human_format

model = BiMambaTranslator(ModelConfig(**cfg['model']))
n = count_parameters(model)
print(f'Bi-Mamba parameters: {human_format(n)}  ({n:,})')


## 8. Train end-to-end

Checkpoint sẽ được ghi vào `runs/bi_mamba_55m/`. Có thể dừng và bật lại notebook với `--resume <ckpt>` để train tiếp.

In [ ]:
!python scripts/train.py --config configs/_colab.yaml


## 8b. Average + EMA của các checkpoint cuối (boost BLEU "miễn phí")

Hai trick chuẩn của NMT để ép thêm BLEU mà không tốn thêm train time:

* **EMA**: trainer đã maintain Exponential-Moving-Average của weights trong suốt quá trình train (`runs/.../latest_ema.pt`) — đánh giá bằng EMA luôn nhỉnh hơn weights ở step cuối ~0.3–1 BLEU.
* **Checkpoint averaging**: lấy trung bình weights của 5 checkpoint cuối → smooth out high-frequency noise → +0.3–0.8 BLEU.

Kết hợp cả hai (`avg_last5_ema.pt`) thường là điểm tốt nhất.


In [ ]:
# Average last 5 model checkpoints, and last 5 EMA checkpoints, separately.
!python scripts/avg_ckpts.py --ckpts-dir runs/bi_mamba_55m --n 5
!python scripts/avg_ckpts.py --ckpts-dir runs/bi_mamba_55m --n 5 --ema
!ls -lh runs/bi_mamba_55m/*.pt | tail -10


## 9. Đánh giá SacreBLEU + chrF (cả hai chiều zh→vi và vi→zh)

In [ ]:
# Compare 6 versions of the model on the test set:
#   (1) latest model weights        — last training step
#   (2) latest EMA weights          — EMA at last step
#   (3) best model weights          — step with lowest val_loss
#   (4) best EMA weights            — step with lowest ema_val_loss
#   (5) avg of last 5 model ckpts   — Polyak averaging
#   (6) avg of last 5 EMA ckpts     — usually the strongest combination
import os
for label, ckpt in [
    ('latest',         'runs/bi_mamba_55m/latest.pt'),
    ('latest_ema',     'runs/bi_mamba_55m/latest_ema.pt'),
    ('best',           'runs/bi_mamba_55m/best.pt'),
    ('best_ema',       'runs/bi_mamba_55m/best_ema.pt'),
    ('avg_last5',      'runs/bi_mamba_55m/avg_last5.pt'),
    ('avg_last5_ema',  'runs/bi_mamba_55m/avg_last5_ema.pt'),
]:
    if not os.path.exists(ckpt):
        print(f'-- skip {label} (no file: {ckpt})'); continue
    print(f'\n=== {label} ===')
    !python scripts/evaluate.py --config configs/_colab.yaml \
        --checkpoint {ckpt} --num-samples 1000


## 10. Demo dịch

In [ ]:
import os
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.tokenizer import Tokenizer
from bi_mamba_mt.translate import translate_batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_candidates = [
    f"{cfg['train']['output_dir']}/avg_last5_ema.pt",
    f"{cfg['train']['output_dir']}/best_ema.pt",
    f"{cfg['train']['output_dir']}/best.pt",
    f"{cfg['train']['output_dir']}/latest.pt",
]
ckpt_path = next(p for p in _candidates if os.path.exists(p))
print('Loading', ckpt_path)
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model = BiMambaTranslator(ModelConfig(**ckpt['model_cfg'])).to(device)
model.load_state_dict(ckpt['model'], strict=False); model.eval()
tok = Tokenizer(f"{cfg['data']['tokenizer_dir']}/spm.model")

zh_samples = [
    '你好，世界。',
    '今天的天气很好。',
    '这本书我已经读完了。',
]
vi_samples = [
    'Xin chào thế giới.',
    'Hôm nay trời thật đẹp.',
    'Tôi đã đọc xong cuốn sách này.',
]

print('--- zh → vi ---')
for s, t in zip(zh_samples,
                translate_batch(model, tok, zh_samples, 'zh2vi',
                                beam_size=4, device=device)):
    print(f'  {s!r} -> {t!r}')

print('--- vi → zh ---')
for s, t in zip(vi_samples,
                translate_batch(model, tok, vi_samples, 'vi2zh',
                                beam_size=4, device=device)):
    print(f'  {s!r} -> {t!r}')


## 11. (Tuỳ chọn) Lưu checkpoint vào Drive

In [ ]:
import shutil, os
DRIVE_DIR = '/content/drive/MyDrive/bi_mamba_55m'
if os.path.isdir('/content/drive/MyDrive'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for fname in ('latest.pt', 'final.pt', 'config.yaml'):
        src = f"{cfg['train']['output_dir']}/{fname}"
        if os.path.isfile(src):
            shutil.copy(src, DRIVE_DIR)
            print('Copied', src, '->', DRIVE_DIR)
    tok_dst = f'{DRIVE_DIR}/tokenizer'
    os.makedirs(tok_dst, exist_ok=True)
    for fname in ('spm.model', 'spm.vocab'):
        src = f"{cfg['data']['tokenizer_dir']}/{fname}"
        if os.path.isfile(src):
            shutil.copy(src, tok_dst)
    print('Done. Files in', DRIVE_DIR)
else:
    print('Drive not mounted; skipping.')
